# Fine-tuning Florence-2 for handwriting

This notebook fine-tunes `Florence-2-base-ft` on a single writer's English handwriting using the `<OCR>` task prompt. It runs top to bottom on a free Colab T4 and is self-sufficient. Before running, select a GPU runtime (Runtime, Change runtime type, T4 GPU). We deliberately stay on the stock architecture and base size (the fine-tuned model is later exported to Apple Core AI for on-device inference, so no custom layers or exotic ops).

In [ ]:
!pip install -q transformers accelerate timm einops jiwer datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_DIR = '/content/drive/MyDrive/handwriting'     # image folder root
CSV_PATH = f'{DATA_DIR}/labels.csv'                 # columns: image,text[,split]
OUTPUT_DIR = '/content/drive/MyDrive/florence2-hw'  # fine-tuned model destination
MODEL_ID = 'microsoft/Florence-2-base-ft'
TASK_PROMPT = '<OCR>'

In [ ]:
FREEZE_VISION = True              # train language head only (single writer, small data)
EPOCHS, LEARNING_RATE = 6, 5e-6
BATCH_SIZE, GRAD_ACCUM = 2, 8     # effective batch = 16
MAX_TARGET_LEN = 512             # truncate long transcriptions
VAL_FRACTION = 0.1               # used only when the CSV has no split column

## Configuration

Edit the two cells above before running.
- `DATA_DIR` / `CSV_PATH` point at your Drive (CSV columns `image,text`, optional `split`).
- `FREEZE_VISION` trains the language head only (leave on for small data).
- Effective batch is `BATCH_SIZE * GRAD_ACCUM` (raise `GRAD_ACCUM` first if you hit OOM).
- Try first: more `EPOCHS` (6 to 12) before touching `LEARNING_RATE`.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True, attn_implementation='eager').to('cuda')

## Why base, and why freeze the vision encoder

The base size keeps memory and Core AI export simple (small, stock, on-device friendly). With a single writer and few pages, the pretrained vision encoder already reads strokes well, so we freeze it and adapt only the language head. This cuts trainable parameters, lowers memory, and resists overfitting on tiny data. Unfreeze later once more pages are labelled (see the pointers at the end).

In [ ]:
import pandas as pd
df = pd.read_csv(CSV_PATH)
if 'split' not in df.columns:
    val = df.sample(frac=VAL_FRACTION, random_state=42).index
    df['split'] = ['val' if i in val else 'train' for i in df.index]

In [ ]:
df['split'].value_counts()

## Canonical list representation

Handwritten lists use mixed markers (•, –, *, ·). We map them all to one canonical marker (`- `), one bullet per line, with a fixed nesting rule (two spaces per level). A single representation is easier to learn, tokenizer-friendly, and doubles as valid Markdown downstream. Prose lines are left untouched. Tune `BULLET_MARKER` / `INDENT_WIDTH` below.

In [ ]:
BULLET_MARKER = '- '   # canonical marker (tokenizer-friendly markdown)
INDENT_WIDTH = 2       # spaces per nesting level
LINE_SEP = '\n'        # bullet separator (switch to a sentinel if the round-trip check fails)

In [ ]:
def _norm_line(r):
    s = r.lstrip()
    if s[:1] in '•–*·-':
        return ' ' * (INDENT_WIDTH * ((len(r) - len(s)) // INDENT_WIDTH)) + BULLET_MARKER + s[1:].strip()
    return r.rstrip()

In [ ]:
def normalise(text):
    return LINE_SEP.join(_norm_line(r) for r in str(text).splitlines() if r.strip())

## Newline fidelity

The target must survive tokenisation, so we round-trip a sample and assert the newline and marker come back intact (Florence-2 uses a byte-level BPE, which normally preserves both). If the assert fails, set `LINE_SEP` to a rare tokenizer-preserved sentinel (e.g. ` <nl> `), keep `normalise` joining on it, and convert it back to `\n` at decode time.

In [ ]:
_t = normalise('• alpha\n  – beta\n* gamma')
_ids = processor.tokenizer(_t, return_tensors='pt').input_ids
_back = processor.tokenizer.decode(_ids[0], skip_special_tokens=True)
assert LINE_SEP in _back, 'newline lost in round-trip; set LINE_SEP to a sentinel (see note)'
assert BULLET_MARKER in _back, 'marker lost in round-trip'

In [ ]:
from PIL import Image
from torch.utils.data import Dataset

In [ ]:
class HWDataset(Dataset):
    def __init__(self, frame): self.rows = frame.reset_index(drop=True)
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): r = self.rows.iloc[i]; return Image.open(f'{DATA_DIR}/{r.image}').convert('RGB'), TASK_PROMPT, normalise(r.text)

In [ ]:
train_ds = HWDataset(df[df.split == 'train'])
val_ds = HWDataset(df[df.split == 'val'])
len(train_ds), len(val_ds)

In [ ]:
def collate(batch):
    images, prompts, texts = zip(*batch)
    enc = processor(text=list(prompts), images=list(images), return_tensors='pt', padding=True)
    labels = processor.tokenizer(list(texts), return_tensors='pt', padding=True, truncation=True, max_length=MAX_TARGET_LEN).input_ids
    enc['labels'] = labels.masked_fill(labels == processor.tokenizer.pad_token_id, -100); return enc

In [ ]:
if FREEZE_VISION:
    for p in model.vision_tower.parameters(): p.requires_grad = False
sum(p.numel() for p in model.parameters() if p.requires_grad)

## Training

We use `Trainer` for the loss loop and evaluate loss each epoch. Mixed precision is fp16 (the T4 has no bf16). Memory levers, in order: raise `GRAD_ACCUM` (keeps the effective batch, uses less memory), lower `BATCH_SIZE`, shorten `MAX_TARGET_LEN`, then consider LoRA. CER and WER are computed separately below via beam-search generation (more faithful than argmax over logits).

In [ ]:
from transformers import TrainingArguments, Trainer
args = TrainingArguments(OUTPUT_DIR, per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE, num_train_epochs=EPOCHS, fp16=True, eval_strategy='epoch', save_strategy='epoch',
    logging_steps=10, remove_unused_columns=False, report_to='none', label_names=['labels'])

In [ ]:
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)

In [ ]:
trainer.train()

## CER vs WER

CER (character error rate) is edit distance over characters; WER over words. For handwriting, CER is the gentler, more informative signal (a single misread letter does not fail an entire word). Watch both, but optimise toward CER early on. Lower is better (0 is perfect).

In [ ]:
import jiwer
from tqdm.auto import tqdm
model.eval();

## Inference decoding

We decode generations with the raw tokenizer (`skip_special_tokens=True`), not `post_process_generation(..., task='<OCR>')`. The OCR post-processor can collapse whitespace and flatten line structure, erasing our bullets and newlines. Raw decoding preserves the canonical layout we trained on.

In [ ]:
def transcribe(img):
    enc = processor(text=TASK_PROMPT, images=img, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=enc['input_ids'], pixel_values=enc['pixel_values'], max_new_tokens=MAX_TARGET_LEN, num_beams=3)
    return processor.tokenizer.decode(out[0], skip_special_tokens=True).strip()

In [ ]:
val_df = df[df.split == 'val'].reset_index(drop=True)
preds = [transcribe(Image.open(f'{DATA_DIR}/{r.image}').convert('RGB')) for r in tqdm(list(val_df.itertuples()))]
refs = [normalise(t) for t in val_df.text]

In [ ]:
{'CER': jiwer.cer(refs, preds), 'WER': jiwer.wer(refs, preds)}

## Structural metric

CER and WER score characters and words but are blind to layout (a list flattened onto one line can still score well). On the list subset of the held-out split, we therefore report marker precision and recall (correctly placed `- ` lines), separately from CER/WER, so layout stays measurable.

In [ ]:
is_list = [any(l.startswith(BULLET_MARKER) for l in r.splitlines()) for r in refs]
lp = [p for p, f in zip(preds, is_list) if f]
lr = [r for r, f in zip(refs, is_list) if f]

In [ ]:
def marker_lines(t): return {i for i, l in enumerate(t.splitlines()) if l.startswith(BULLET_MARKER)}
tp = sum(len(marker_lines(p) & marker_lines(r)) for p, r in zip(lp, lr))
pp = sum(len(marker_lines(p)) for p in lp); rr = sum(len(marker_lines(r)) for r in lr)
{'list_pages': len(lr), 'marker_precision': tp / max(pp, 1), 'marker_recall': tp / max(rr, 1)}

In [ ]:
from IPython.display import display
row = val_df.iloc[0]
sample = Image.open(f'{DATA_DIR}/{row.image}').convert('RGB')
display(sample)
{'prediction': transcribe(sample), 'ground_truth': normalise(row.text)}

In [ ]:
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

## Where to go next

- Move `base` to `large` for accuracy if memory allows.
- Unfreeze the vision encoder (`FREEZE_VISION = False`) once more pages are labelled.
- LoRA via PEFT to cut memory and speed iteration.
- Tune learning rate, epochs, and effective batch size (grad accumulation).
- Mild augmentation only (slight contrast, small rotation); aggressive transforms distort handwriting and hurt accuracy.
- Bootstrap more labels with a stronger model (e.g. `qwen3-vl`), then correct by hand.
- Aim for roughly 20% list-containing pages, mirrored in the test split, so list performance stays measurable.

Core AI export (`torch.export` to `coreai-torch`) lives in a separate notebook (this one stays focused on training).